In [ ]:
!pip install -q transformers peft bitsandbytes accelerate datasets scikit-learn

import json, random
from collections import Counter
import torch
from torch.utils.data import Dataset as TorchDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID       = "unsloth/Llama-3.2-3B-Instruct"
JSON_PATH      = "merged_math_questions.json"
LORA_SAVE_PATH = "level_predictor_lora"
MAX_SEQ_LEN    = 128
EPOCHS         = 3
LR             = 2e-4

with open(JSON_PATH) as f:
    data = json.load(f)
lvls = data["levels"]
samples = []
for sub in ["addition", "subtraction"]:
    for q in lvls["level_1"]["subtypes"][sub]["questions"]:
        samples.append({"question": q["question"], "level": 1})
for q in lvls["level_2"]["questions"]:
    samples.append({"question": q["question"], "level": 2})
for q in lvls["level_3"]["questions"]:
    samples.append({"question": q["question"], "level": 3})
random.seed(42)
random.shuffle(samples)
print(f"Total: {len(samples)} | {dict(Counter(s['level'] for s in samples))}")

train_s, temp = train_test_split(samples, test_size=0.2, random_state=42, stratify=[s["level"] for s in samples])
val_s, test_s = train_test_split(temp,    test_size=0.5, random_state=42, stratify=[s["level"] for s in temp])
print(f"Train {len(train_s)} | Val {len(val_s)} | Test {len(test_s)}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def make_prompt(question, level=None):
    system = (
        "You are a math difficulty classifier. "
        "Given a math word problem, reply with ONLY the level number:\n"
        "1 = Addition or Subtraction\n"
        "2 = Multiplication\n"
        "3 = Division"
    )
    prompt = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n\n{system}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\nClassify this problem:\n{question}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    if level is not None:
        prompt += f"{level}<|eot_id|>"
    return prompt

class MathDataset(TorchDataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        encoded = tokenizer(
            make_prompt(s["question"], s["level"]),
            truncation=True,
            max_length=MAX_SEQ_LEN,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids":      encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels":         encoded["input_ids"].squeeze(0).clone(),
        }

train_ds = MathDataset(train_s)
val_ds   = MathDataset(val_s)

training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=4,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    weight_decay=0.01,
    fp16=True,
    logging_steps=50,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=42,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

trainer.train()

model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)
print(f"Saved to {LORA_SAVE_PATH}/")

model.eval()

def predict(question):
    prompt = make_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=3, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    for ch in text:
        if ch in ("1", "2", "3"):
            return int(ch)
    return 1

y_true, y_pred = [], []
for s in test_s:
    y_true.append(s["level"])
    y_pred.append(predict(s["question"]))

print(classification_report(y_true, y_pred, target_names=["Level 1", "Level 2", "Level 3"]))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
print(f"Accuracy: {sum(t==p for t,p in zip(y_true,y_pred))/len(y_true)*100:.2f}%")

Total: 4059 | {2: 1212, 1: 1971, 3: 876}
Train 3247 | Val 406 | Test 406


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

trainable params: 12,156,928 || all params: 3,224,906,752 || trainable%: 0.3770


Step,Training Loss,Validation Loss
100,0.333894,0.263463


Step,Training Loss,Validation Loss
100,0.333894,0.263463
200,0.246676,0.248276
